### File Reader (LAMMPS)

- Read all *dump. files from LAMMPS saved in 'pour' folders 

- Save all particles data in a python dictionary as frames

In [ ]:
import glob
import torch
import numpy as np
import os


def get_avg_coordNum(file_path):
    """Get average coordination number from the output file."""
    last_value = None
    
    with open(file_path, 'r') as f:
        for line in f:
            # Check if line starts with a number (timestep line)
            if line.strip() and line.split()[0].isdigit():
                # Get the last column value
                columns = line.split()
                last_value = columns[-1]
    coordination_number = float(last_value)
    return coordination_number

def read_radius_from_config(config_file):
    """
    Read LAMMPS config file and extract radius from diameter column
    Returns array of radius values ordered by atom ID
    """
    with open(config_file, 'r') as f:
        lines = f.readlines()
    
    # Find Atoms section
    particles_start = None
    for i, line in enumerate(lines):
        if 'Atoms' in line:
            particles_start = i + 2  # Skip "Atoms" line and blank line
            break
    
    if particles_start is None:
        raise ValueError("Atoms section not found in config file")
    
    # Extract ID and diameter
    particle_data = []
    for line in lines[particles_start:]:
        if not line.strip():  # Stop at blank line
            break
        parts = line.split()
        if len(parts) >= 3:
            particle_id = int(parts[0])
            diameter = float(parts[2])  # dia is 3rd column
            particle_data.append((particle_id, diameter))
    
    # Sort by particle ID to ensure correct order
    particle_data.sort(key=lambda x: x[0])
    
    # Extract diameters and convert to radius
    diameters = np.array([d for _, d in particle_data])
    radius = diameters / 2.0
    
    return radius

def read_dump_file(file_path):
    """
    Read a LAMMPS dump file and extract particle data
    Returns a dictionary with keys: 'N', 'dim', 'x', 'v', 'type', etc.
    """
    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    frame_data = {}
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if line.startswith("ITEM: TIMESTEP"):
            i += 1
            frame_data['timestep'] = int(lines[i].strip())
        elif line.startswith("ITEM: NUMBER OF ATOMS"):
            i += 1
            frame_data['N'] = int(lines[i].strip())
        elif line.startswith("ITEM: BOX BOUNDS"):
            i += 1
            bounds = []
            for _ in range(3):
                bounds.append(list(map(float, lines[i].strip().split())))
                i += 1
            frame_data['box_bounds'] = np.array(bounds)
            frame_data['dim'] = 3  # Assuming 3D
            continue  # Skip incrementing i here
        elif line.startswith("ITEM: ATOMS"):
            headers = line.split()[2:]  # Get column headers
            data = []
            for j in range(frame_data['N']):
                i += 1
                parts = lines[i].strip().split()
                data.append([float(part) for part in parts])
            data_array = np.array(data)
            
            # Map headers to data columns
            for idx, header in enumerate(headers):
                frame_data[header] = data_array[:, idx]
        i += 1
    
    return frame_data

def read_all_frames(folder_path, pattern="dump.stress.*", max_frames=1000000000):
    """
    Read all dump files in the specified folder matching the pattern
    Returns a list of frames (dictionaries)
    """
    file_paths = sorted(glob.glob(os.path.join(folder_path, pattern)))
    frames = []
    for file_path in file_paths[:max_frames]:
        frame = read_dump_file(file_path)
        frames.append(frame)
    return frames
# ====================================================================================================================
# ====================================================================================================================
frames = read_all_frames(r"F:\DEM_DATA\const_V_3D\578_085\pour")
for i in range(len(frames)):
    radius = read_radius_from_config(config_file = r"F:\DEM_DATA\const_V_3D\578_085\config_578.txt")
    frames[i]['radius'] = radius
#====================================================================================================================
frames_sorted = sorted(frames, key=lambda x: int(x['timestep']))

In [ ]:
"""
phi_voxel_gaussian.py
=====================
Two-stage local solid volume fraction for DEM granular data.

Device strategy
---------------
  cKDTree        : always CPU (scipy limitation — used only for candidate lookup)
  tensor math    : GPU if available, else CPU
  Rule           : every tensor is created/moved via the module-level DEVICE;
                   .cpu().numpy() is called only when passing to cKDTree.

Gridding
--------
  Full box  : Ly = 0.40 m  (Ny_full = 40 at d = 0.01 m)
  Bulk range: y ∈ [y_lo+d, y_hi−d]  →  Ly_eff = 0.38 m
  Grid      : Nx=50, Ny=38, Nz=10
  PBC       : x (flow), z (vorticity)  →  circular padding
  Walls     : y (gradient)             →  reflect padding
"""

import numpy as np
import torch
import torch.nn.functional as F
from scipy.spatial import cKDTree
from typing import Tuple, Optional, List

# ── global device ─────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[device]  using {DEVICE}")


# ─────────────────────────────────────────────────────────────────────────────
# 0.  TRIMMED BOUNDS
# ─────────────────────────────────────────────────────────────────────────────

def get_bulk_bounds(box_bounds: np.ndarray, particle_diam: float = 0.01) -> torch.Tensor:
    """
    Trim one particle diameter from each y-wall; keep x, z unchanged.
    Returns (3,2) float64 tensor on DEVICE.
    """
    b    = np.asarray(box_bounds, dtype=np.float64)
    bulk = b.copy()
    bulk[1, 0] += particle_diam
    bulk[1, 1] -= particle_diam
    t = torch.from_numpy(bulk).to(DEVICE)
    print(f"[bounds]  y_full=[{b[1,0]:.4f},{b[1,1]:.4f}]  "
          f"y_bulk=[{bulk[1,0]:.4f},{bulk[1,1]:.4f}]  "
          f"Ly_eff={bulk[1,1]-bulk[1,0]:.4f} m")
    return t


# ─────────────────────────────────────────────────────────────────────────────
# 1.  PARTICLE ARRAYS
# ─────────────────────────────────────────────────────────────────────────────

def build_particle_arrays(
        snapshot:  dict,
        bulk_type: int = 3,
) -> Tuple[torch.Tensor, torch.Tensor, cKDTree]:
    """
    Returns
    -------
    centers : (Np,3) float64 on DEVICE
    radii   : (Np,)  float64 on DEVICE
    tree    : cKDTree built on CPU numpy array (scipy requirement)
    """
    type_arr = np.asarray(snapshot['type'])
    mask     = (type_arr == bulk_type)

    centers_np = np.column_stack([
        np.asarray(snapshot['x'],      dtype=np.float64)[mask],
        np.asarray(snapshot['y'],      dtype=np.float64)[mask],
        np.asarray(snapshot['z'],      dtype=np.float64)[mask],
    ])                                                        # (Np,3) CPU numpy
    radii_np   = np.asarray(snapshot['radius'], dtype=np.float64)[mask]

    centers = torch.from_numpy(centers_np).to(DEVICE)        # GPU
    radii   = torch.from_numpy(radii_np  ).to(DEVICE)
    tree    = cKDTree(centers_np)                             # CPU only

    print(f"[build]  Np={len(radii_np)}  "
          f"r_mean={radii_np.mean():.5f}  r_max={radii_np.max():.5f}")
    return centers, radii, tree


# ─────────────────────────────────────────────────────────────────────────────
# 2.  MAX OVERLAP → POINT SEPARATION
# ─────────────────────────────────────────────────────────────────────────────

def compute_max_overlap_and_separation(
        centers:  torch.Tensor,   # DEVICE
        radii:    torch.Tensor,   # DEVICE
        tree:     cKDTree,        # CPU
        n_sample: int   = 2000,
        safety:   float = 1.05,
) -> Tuple[float, float]:
    """
    Samples n_sample particles, queries CPU tree, computes distances on GPU.
    """
    n_total  = centers.shape[0]
    n_sample = min(n_sample, n_total)
    idx      = torch.randperm(n_total, generator=torch.Generator().manual_seed(42))[:n_sample]

    # one-time CPU copy for tree queries
    centers_cpu = centers.cpu().numpy()
    r_max_val   = radii.max().item()
    max_overlap = torch.tensor(0.0, dtype=centers.dtype, device=DEVICE)

    for i in idx.tolist():
        r_i  = radii[i]
        nbrs = tree.query_ball_point(centers_cpu[i], r_i.item() + r_max_val)
        nbrs = [j for j in nbrs if j != i]
        if not nbrs:
            continue
        j_idx    = torch.tensor(nbrs, dtype=torch.long, device=DEVICE)
        dists    = torch.linalg.norm(centers[i] - centers[j_idx], dim=1)  # GPU
        overlaps = r_i + radii[j_idx] - dists
        best     = overlaps.max()
        if best > max_overlap:
            max_overlap = best

    lens_radius = max_overlap / 2.0
    separation  = safety * torch.max(lens_radius, radii.max() * 0.05)
    print(f"[overlap]  max_overlap={max_overlap.item():.5f} m  "
          f"sep={separation.item():.5f} m")
    return max_overlap.item(), separation.item()


# ─────────────────────────────────────────────────────────────────────────────
# 3.  TEMPLATE VOXEL POINTS
# ─────────────────────────────────────────────────────────────────────────────

def make_template_voxel_points(
        dx: float, dy: float, dz: float,
        separation: float,
) -> Tuple[torch.Tensor, int]:
    """
    Sub-voxel uniform points in [0,dx]×[0,dy]×[0,dz]. Built once, on DEVICE.
    """
    nx_pt = max(1, int(np.floor(dx / separation)))
    ny_pt = max(1, int(np.floor(dy / separation)))
    nz_pt = max(1, int(np.floor(dz / separation)))

    xs = np.linspace(dx/(2*nx_pt), dx - dx/(2*nx_pt), nx_pt)
    ys = np.linspace(dy/(2*ny_pt), dy - dy/(2*ny_pt), ny_pt)
    zs = np.linspace(dz/(2*nz_pt), dz - dz/(2*nz_pt), nz_pt)

    XX, YY, ZZ   = np.meshgrid(xs, ys, zs, indexing='ij')
    pts_np       = np.column_stack([XX.ravel(), YY.ravel(), ZZ.ravel()])
    template_pts = torch.from_numpy(pts_np).to(DEVICE)       # GPU
    Npt          = len(template_pts)

    print(f"[template]  {nx_pt}×{ny_pt}×{nz_pt} = {Npt} pts/voxel  "
          f"voxel=({dx:.4f},{dy:.4f},{dz:.4f})")
    return template_pts, Npt


# ─────────────────────────────────────────────────────────────────────────────
# 4.  SINGLE-VOXEL PHI  (GPU math, CPU tree query)
# ─────────────────────────────────────────────────────────────────────────────

def _phi_one_voxel(
        vox_origin:    torch.Tensor,   # (3,)    DEVICE
        template_pts:  torch.Tensor,   # (Npt,3) DEVICE
        centers:       torch.Tensor,   # (Np,3)  DEVICE
        centers_cpu:   np.ndarray,     # (Np,3)  CPU — tree queries only
        radii:         torch.Tensor,   # (Np,)   DEVICE
        tree:          cKDTree,
        r_max:         float,
        vox_half_diag: float,
) -> float:
    """OR-test point counting. Candidate lookup on CPU tree, distance math on GPU."""
    pts        = vox_origin + template_pts                              # (Npt,3) GPU
    vox_ctr    = (vox_origin + template_pts.mean(dim=0)).cpu().numpy() # CPU for tree
    candidates = tree.query_ball_point(vox_ctr, vox_half_diag + r_max)

    if not candidates:
        return 0.0

    cand_idx   = torch.tensor(candidates, dtype=torch.long, device=DEVICE)
    dists      = torch.cdist(pts, centers[cand_idx], p=2)              # (Npt,Nc) GPU
    inside_any = (dists <= radii[cand_idx]).any(dim=1)
    return inside_any.float().mean().item()

# ─── PBC FUNCTION ──────────────────────────────────────────────────────────────

def _augment_pbc_images(
        centers:     torch.Tensor,   # (Np,3) DEVICE
        radii:       torch.Tensor,   # (Np,)  DEVICE
        bulk_bounds: torch.Tensor,   # (3,2)  DEVICE
        cutoff:      float,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Add image particles for PBC in x (dim 0) and z (dim 2).
    Only duplicates particles within `cutoff` of each x/z boundary face.
    y is bounded (walls) — no images needed.
    """
    x_lo, x_hi = bulk_bounds[0, 0].item(), bulk_bounds[0, 1].item()
    z_lo, z_hi = bulk_bounds[2, 0].item(), bulk_bounds[2, 1].item()
    Lx = x_hi - x_lo
    Lz = z_hi - z_lo

    cx, cz = centers[:, 0], centers[:, 2]

    near_xlo = (cx - x_lo) < cutoff
    near_xhi = (x_hi - cx) < cutoff
    near_zlo = (cz - z_lo) < cutoff
    near_zhi = (z_hi - cz) < cutoff

    # 4 face images + 4 corner images
    shifts = [
        ( Lx,   0.0,  near_xlo),
        (-Lx,   0.0,  near_xhi),
        ( 0.0,  Lz,   near_zlo),
        ( 0.0, -Lz,   near_zhi),
        ( Lx,   Lz,   near_xlo & near_zlo),
        ( Lx,  -Lz,   near_xlo & near_zhi),
        (-Lx,   Lz,   near_xhi & near_zlo),
        (-Lx,  -Lz,   near_xhi & near_zhi),
    ]

    aug_c = [centers]
    aug_r = [radii]
    for dx, dz, mask in shifts:
        if mask.any():
            c = centers[mask].clone()
            c[:, 0] += dx
            c[:, 2] += dz
            aug_c.append(c)
            aug_r.append(radii[mask])

    centers_aug = torch.cat(aug_c, dim=0)
    radii_aug   = torch.cat(aug_r, dim=0)
    n_img = centers_aug.shape[0] - centers.shape[0]
    print(f"[pbc images]  added {n_img} image particles  "
          f"(total {centers_aug.shape[0]} incl. images)")
    return centers_aug, radii_aug
# ─────────────────────────────────────────────────────────────────────────────
# 5.  FULL VOXEL SWEEP
# ─────────────────────────────────────────────────────────────────────────────

# ─── PATCHED compute_phi_voxelwise ───────────────────────────────────────────
# ** augment particles with PBC images BEFORE the sweep,
# then rebuild the KD-tree from the augmented set.

def compute_phi_voxelwise(
        centers:     torch.Tensor,
        radii:       torch.Tensor,
        tree:        cKDTree,           # original tree (used nowhere below now)
        bulk_bounds: torch.Tensor,
        grid_shape:  Tuple[int, int, int],
        separation:  float,
        verbose:     bool = True,
) -> torch.Tensor:
    """Returns phi_voxel (Nx,Ny,Nz) float64 on DEVICE."""
    Nx, Ny, Nz  = grid_shape
    diffs       = bulk_bounds[:, 1] - bulk_bounds[:, 0]
    Lx, Ly, Lz = diffs[0].item(), diffs[1].item(), diffs[2].item()
    dx, dy, dz  = Lx/Nx, Ly/Ny, Lz/Nz

    vox_half_diag = 0.5 * (dx**2 + dy**2 + dz**2)**0.5
    r_max         = radii.max().item()
    cutoff        = r_max + vox_half_diag   # query radius used in _phi_one_voxel

    # ── PBC IMAGE AUGMENTATION (new) ─────────────────────────────────────────
    centers_aug, radii_aug = _augment_pbc_images(
        centers, radii, bulk_bounds, cutoff
    )
    centers_cpu = centers_aug.cpu().numpy()      # rebuild tree on augmented set
    tree_aug    = cKDTree(centers_cpu)
    # ─────────────────────────────────────────────────────────────────────────

    template_pts, _ = make_template_voxel_points(dx, dy, dz, separation)

    phi_voxel  = torch.zeros((Nx, Ny, Nz), dtype=torch.float64, device=DEVICE)
    x0, y0, z0 = bulk_bounds[:, 0].tolist()
    total, done = Nx * Ny * Nz, 0

    for ix in range(Nx):
        for iy in range(Ny):
            for iz in range(Nz):
                origin = torch.tensor(
                    [x0 + ix*dx, y0 + iy*dy, z0 + iz*dz],
                    dtype=torch.float64, device=DEVICE,
                )
                phi_voxel[ix, iy, iz] = _phi_one_voxel(
                    origin, template_pts,
                    centers_aug, centers_cpu,   # ← augmented set
                    radii_aug, tree_aug,         # ← augmented tree
                    r_max, vox_half_diag,
                )
                done += 1
                if verbose and done % max(1, total // 20) == 0:
                    filled = phi_voxel[phi_voxel > 0]
                    mean_s = f"phi_mean={filled.mean().item():.4f}" if filled.numel() else "phi_mean=n/a"
                    print(f"[voxel]  {done}/{total} ({100*done//total}%)  {mean_s}")

    if verbose:
        print(f"[voxel done]  mean={phi_voxel.mean():.4f}  "
              f"min={phi_voxel.min():.4f}  max={phi_voxel.max():.4f}")
    return phi_voxel

# ─────────────────────────────────────────────────────────────────────────────
# 6.  GAUSSIAN SMOOTHING
# ─────────────────────────────────────────────────────────────────────────────

def gaussian_smooth_phi(
        phi_voxel:   torch.Tensor,
        bulk_bounds: torch.Tensor,
        grid_shape:  Tuple[int, int, int],
        sigma_phys:  Optional[float] = None,
        sigma_vox:   Optional[Tuple[float, float, float]] = None,
        truncate:    float = 3.0,
) -> torch.Tensor:
    """
    Separable Gaussian filter entirely on DEVICE.
      x → circular (PBC)   y → reflect (walls)   z → circular (PBC)
    """
    Nx, Ny, Nz = grid_shape
    diffs = bulk_bounds[:, 1] - bulk_bounds[:, 0]
    dx    = (diffs[0] / Nx).item()
    dy    = (diffs[1] / Ny).item()
    dz    = (diffs[2] / Nz).item()

    if sigma_vox is not None:
        sx, sy, sz = sigma_vox
    elif sigma_phys is not None:
        sx, sy, sz = sigma_phys/dx, sigma_phys/dy, sigma_phys/dz
    else:
        sx = sy = sz = 2.0

    print(f"[smooth]  sigma_vox=({sx:.2f},{sy:.2f},{sz:.2f})  "
          f"voxel=({dx:.4f},{dy:.4f},{dz:.4f})")

    def _kernel1d(sigma: float) -> torch.Tensor:
        r = int(truncate * sigma + 0.5)
        x = torch.arange(-r, r+1, dtype=phi_voxel.dtype, device=DEVICE)
        k = torch.exp(-0.5 * (x / sigma)**2)
        return k / k.sum()

    kx, ky, kz = _kernel1d(sx), _kernel1d(sy), _kernel1d(sz)
    out = phi_voxel.unsqueeze(0).unsqueeze(0)   # (1,1,Nx,Ny,Nz)

    # x — circular (PBC)
    px  = len(kx) // 2
    out = F.pad(out, (0, 0, 0, 0, px, px), mode='circular')
    out = F.conv3d(out, kx.view(1, 1, -1, 1, 1))

    # y — reflect (walls)
    py  = len(ky) // 2
    out = F.pad(out, (0, 0, py, py, 0, 0), mode='reflect')
    out = F.conv3d(out, ky.view(1, 1, 1, -1, 1))

    # z — circular (PBC)
    pz  = len(kz) // 2
    out = F.pad(out, (pz, pz, 0, 0, 0, 0), mode='circular')
    out = F.conv3d(out, kz.view(1, 1, 1, 1, -1))

    phi_smooth = out.squeeze()
    print(f"[smooth]  mean={phi_smooth.mean():.4f}  "
          f"min={phi_smooth.min():.4f}  max={phi_smooth.max():.4f}")
    return phi_smooth


# ─────────────────────────────────────────────────────────────────────────────
# 7.  SINGLE-SNAPSHOT PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

def compute_phi_two_stage(
        snapshot:         dict,
        grid_shape:       Tuple[int, int, int] = (50, 38, 10),
        bulk_type:        int   = 3,
        particle_diam:    float = 0.01,
        n_overlap_sample: int   = 2000,
        safety:           float = 1.05,
        sigma_phys:       Optional[float] = None,
        sigma_vox:        Optional[Tuple[float, float, float]] = None,
        verbose:          bool  = True,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Returns
    -------
    phi_voxel  : (Nx,Ny,Nz)       float64  DEVICE
    phi_smooth : (Nx,Ny,Nz)       float64  DEVICE
    phi_tensor : (1,Nx,Ny,Nz,1)   float32  DEVICE  — ML-ready
    """
    centers, radii, tree = build_particle_arrays(snapshot, bulk_type)
    bulk_bounds          = get_bulk_bounds(snapshot['box_bounds'], particle_diam)

    _, separation = compute_max_overlap_and_separation(
        centers, radii, tree, n_sample=n_overlap_sample, safety=safety
    )
    phi_voxel  = compute_phi_voxelwise(
        centers, radii, tree, bulk_bounds, grid_shape, separation, verbose
    )
    phi_smooth = gaussian_smooth_phi(
        phi_voxel, bulk_bounds, grid_shape, sigma_phys, sigma_vox
    )
    phi_tensor = phi_smooth.float().unsqueeze(0).unsqueeze(-1)  # (1,Nx,Ny,Nz,1)
    return phi_voxel, phi_smooth, phi_tensor


# ─────────────────────────────────────────────────────────────────────────────
# 8.  TIME-SERIES WRAPPER
# ─────────────────────────────────────────────────────────────────────────────

def compute_phi_timeseries(
        frames:        List[dict],
        grid_shape:    Tuple[int, int, int] = (50, 38, 10),
        bulk_type:     int   = 3,
        particle_diam: float = 0.01,
        safety:        float = 1.05,
        sigma_phys:    Optional[float] = None,
        sigma_vox:     Optional[Tuple[float, float, float]] = None,
        verbose:       bool  = True,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Returns
    -------
    phi_voxel_ts  : (Nt,Nx,Ny,Nz)    float32  DEVICE
    phi_smooth_ts : (Nt,Nx,Ny,Nz,1)  float32  DEVICE
    """
    Nt         = len(frames)
    Nx, Ny, Nz = grid_shape

    # pre-compute from frame 0 (radii and box are constant)
    cen0, rad0, tree0 = build_particle_arrays(frames[0], bulk_type)
    bulk_bounds       = get_bulk_bounds(frames[0]['box_bounds'], particle_diam)
    _, separation     = compute_max_overlap_and_separation(
        cen0, rad0, tree0, n_sample=2000, safety=safety
    )
    diffs = bulk_bounds[:, 1] - bulk_bounds[:, 0]
    dx, dy, dz = (diffs[0]/Nx).item(), (diffs[1]/Ny).item(), (diffs[2]/Nz).item()
    _, Npt = make_template_voxel_points(dx, dy, dz, separation)
    print(f"\n[timeseries]  Nt={Nt}  grid={Nx}×{Ny}×{Nz}  "
          f"Ly_eff={diffs[1].item():.4f} m  Npt/voxel={Npt}  device={DEVICE}")

    phi_v_all = torch.zeros((Nt, Nx, Ny, Nz), dtype=torch.float32, device=DEVICE)
    phi_s_all = torch.zeros((Nt, Nx, Ny, Nz), dtype=torch.float32, device=DEVICE)

    for t, frame in enumerate(frames):
        print(f"\n[frame {t+1}/{Nt}]  timestep={frame['timestep']}")
        cen, rad, tr = build_particle_arrays(frame, bulk_type)

        phi_v = compute_phi_voxelwise(
            cen, rad, tr, bulk_bounds, grid_shape, separation, verbose=False
        )
        phi_s = gaussian_smooth_phi(
            phi_v, bulk_bounds, grid_shape, sigma_phys, sigma_vox
        )
        phi_v_all[t] = phi_v.float()
        phi_s_all[t] = phi_s.float()
        print(f"  raw:    mean={phi_v.mean():.4f}  min={phi_v.min():.4f}  max={phi_v.max():.4f}")
        print(f"  smooth: mean={phi_s.mean():.4f}  min={phi_s.min():.4f}  max={phi_s.max():.4f}")

    phi_smooth_ts = phi_s_all.unsqueeze(-1)
    print(f"\n[done]  phi_smooth_ts: {list(phi_smooth_ts.shape)}  device={phi_smooth_ts.device}")
    return phi_v_all, phi_smooth_ts


# ─────────────────────────────────────────────────────────────────────────────
# 9.  DIAGNOSTICS
# ─────────────────────────────────────────────────────────────────────────────

def diagnose_phi(
        phi_voxel:   torch.Tensor,   # (Nt,Nx,Ny,Nz)
        phi_smooth:  torch.Tensor,   # (Nt,Nx,Ny,Nz) or (Nt,Nx,Ny,Nz,1)
        bulk_bounds: torch.Tensor,   # (3,2)
) -> None:
    if phi_smooth.dim() == 5:
        phi_smooth = phi_smooth.squeeze(-1)

    Ny   = phi_voxel.shape[2]
    Ly   = (bulk_bounds[1, 1] - bulk_bounds[1, 0]).item()
    dy   = Ly / Ny
    y_lo = bulk_bounds[1, 0].item()
    y_c  = [y_lo + (iy + 0.5) * dy for iy in range(Ny)]

    # pull to CPU for printing
    phi_v_y = phi_voxel.float().mean(dim=(0, 1, 3)).cpu()
    phi_s_y = phi_smooth.float().mean(dim=(0, 1, 3)).cpu()

    print("\n── phi(y) bulk profile  [avg over t, x, z] ──────────────────────")
    print(f"  {'y_centre':>9}  {'raw':>7}  {'smooth':>7}  bar")
    for iy in range(Ny):
        bar = '█' * int(phi_s_y[iy].item() * 40)
        print(f"  y={y_c[iy]:.4f}  {phi_v_y[iy]:.4f}  {phi_s_y[iy]:.4f}  {bar}")
    print("─────────────────────────────────────────────────────────────────")


# ─────────────────────────────────────────────────────────────────────────────
# 10.  USAGE
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    phi_voxel_ts, phi_smooth_ts = compute_phi_timeseries(
        frames        = frames_sorted,
        grid_shape    = (50, 38, 10),
        bulk_type     = 3,
        particle_diam = 0.01,
        safety        = 1.05,
        sigma_phys    = 0.015,
    )
    bulk_bounds = get_bulk_bounds(frames_sorted[0]['box_bounds'], particle_diam=0.01)
    diagnose_phi(phi_voxel_ts, phi_smooth_ts, bulk_bounds)

    # torch.save(phi_smooth_ts.cpu(), r"D:\ml_granular\Sparse Modeling\vf_588_vt_03\phi_vf588vt03.pt")

### Temporal Coarse Graining

In [ ]:
#Temporal Coarse Graining....
import numpy as np

def strain_based_coarse_graining(phi, strain_rate, dump_freq, time_step,
                                  strain_half_window=0.5, overlap_fraction=0.5):
    """
    Temporally coarse-grain the stress tensor using a Gaussian window centred
    at uniformly spaced strain values.

    The window is specified entirely in STRAIN space so that results are
    independent of dump frequency and timestep once those are fixed.

    Parameters
    ----------
    phi : ndarray, shape (N_dumps, N_grid, 3, 3)
    strain_rate   : float   [1/s]
    dump_freq     : int     [timesteps per dump]
    time_step     : float   [s]
    strain_half_window : float
        Half-width of the averaging window in strain units.
        Default = 0.5 (average over Δγ = 1.0, centred on each output point).
        The Gaussian sigma is set to strain_half_window / 2, so that weights
        fall to e^{-2} ≈ 0.14 at the window edges.
    overlap_fraction : float in [0, 1)
        Fraction of the window that adjacent output centres share.
        0 = non-overlapping, 0.5 (default) = 50% overlap.
        For ML training, 0 or at most 0.5 is recommended to limit
        autocorrelation between samples.

    Returns
    -------
    cg_stress  : ndarray, shape (N_out, N_grid, 3, 3)
    out_strains: ndarray, shape (N_out,)   strain value at each output centre
    """

    N_dumps = phi.shape[0]
    dt_dump  = dump_freq * time_step                          # s per dump
    strain_per_dump = dt_dump * strain_rate                   # Δγ per dump

    # Total strain spanned by the simulation
    total_strain = N_dumps * strain_per_dump

    # Gaussian sigma in STRAIN units (not frames)
    #   sigma = half_window / 2  so weights at edges are e^{-2}
    sigma_strain = strain_half_window / 2.0

    # Step between output centres in strain units
    stride_strain = strain_half_window * (1.0 - overlap_fraction)

    print(f"Temporal CG parameters")
    print(f"  dt_dump            = {dt_dump:.4f} s")
    print(f"  strain per dump    = {strain_per_dump:.4f}")
    print(f"  total strain       = {total_strain:.2f}")
    print(f"  window half-width  = {strain_half_window:.3f}  (strain units)")
    print(f"  Gaussian sigma     = {sigma_strain:.3f}  (strain units)")
    print(f"  output stride      = {stride_strain:.3f}  (strain units)")
    print(f"  frames in window   = {strain_half_window / strain_per_dump:.1f}")

    # Output centre strains: from half_window to (total - half_window)
    out_strains = np.arange(strain_half_window,
                            total_strain - strain_half_window + stride_strain,
                            stride_strain)
    print(f"  N output frames    = {len(out_strains)}")

    cg_phi = np.zeros((len(out_strains),) + phi.shape[1:],
                         dtype=np.float64)

    # Strain value at each dump
    dump_strains = np.arange(N_dumps) * strain_per_dump   # shape (N_dumps,)

    for k, gamma_centre in enumerate(out_strains):

        #1. Gaussian weights in STRAIN space 
        delta_gamma = dump_strains - gamma_centre           # shape (N_dumps,)
        weights     = np.exp(-0.5 * (delta_gamma / sigma_strain) ** 2)

        # Zero out frames outside 3-sigma support to avoid negligible contributions
        weights[np.abs(delta_gamma) > 3.0 * sigma_strain] = 0.0

        w_sum = weights.sum()
        if w_sum < 1e-12:
            # No frames in window — copy nearest available frame
            nearest = int(np.round(gamma_centre / strain_per_dump))
            nearest = np.clip(nearest, 0, N_dumps - 1)
            cg_phi[k] = phi[nearest]
            continue

        weights /= w_sum

        # Weighted sum over all contributing dumps
        # tensordot contracts axis 0 of weights (N_dumps,) with axis 0 of
        # phi (N_dumps, N_grid, 3, 3) -> (N_grid, 3, 3)
        cg_phi[k] = np.tensordot(weights, phi, axes=(0, 0))

    print("Temporal CG complete")
    return cg_phi, out_strains

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import re

#==============================================================================
folder = r"F:\DEM_DATA\const_V_3D\vf578vt085\pour"
vt_match = re.search(r'vt(\d+)', folder)
if vt_match:
    vt_value = vt_match.group(1) 
else:
    print(r"No top wall velocity found")
    vt_value = '0'
vt_to_strain_rate = {'014': 0.035, '003': 0.00775, '03': 0.0775, '085': 0.2125, '14': 0.35, '0': 0.0}
strain_rate = vt_to_strain_rate[vt_value]
dump_freq = 1e5
time_step = 5.18e-6
#====================================================================================
phi_smooth_ts = phi_smooth_ts.cpu().numpy()
phi_vf,_ =strain_based_coarse_graining(phi_smooth_ts, strain_rate, dump_freq, time_step)
print(phi_vf.shape)
torch.save(torch.from_numpy(phi_vf), r"D:\ml_granular\Sparse Modeling\vf_578_vt_085\phi_vf578vt085.pt")